# 2.05 蒙特卡洛方法：没有模型，就去"试"

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/02_rl/2.05_monte_carlo.ipynb)


## 为什么要学

前面的算法都需要知道 $p(s'|s,a)$ 和 $p(r|s,a)$。真实世界里没人能告诉你。比如线头插入时，力和形变的物理模型根本写不出来。

**无模型（model-free）** 方法只需要与环境交互得到的数据。最直接的想法是：**状态值是期望回报，那我就多试几次，取平均**。这就是蒙特卡洛（Monte Carlo，MC）方法。对应书中第 5 章。

> 第 8 章的 RECAP 用"整条轨迹的回报"训练价值函数，本质上就是 MC 估计。

---

## 1. MC Basic：把策略迭代里的"模型"换成"采样"

回忆策略迭代的策略改进：$\pi_{k+1}(s)=\arg\max_a q_{\pi_k}(s,a)$。有模型时 $q$ 由公式算出；没有模型时，直接用定义估计：

$$
q_\pi(s,a)=\mathbb{E}[G_t\mid S_t=s,A_t=a]\approx\frac1N\sum_{i=1}^N g^{(i)}(s,a)
$$

从 $(s,a)$ 出发采样 $N$ 条轨迹，取回报的平均。

**MC Basic 算法**：对每轮 $k$：

1. 对每个 $(s,a)$，从它出发按 $\pi_k$ 采样若干条轨迹，平均回报得到 $q_k(s,a)$；
2. $\pi_{k+1}(s) = \arg\max_a q_k(s,a)$。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from vla_ladder.envs import GridWorld, policy_to_matrix

env = GridWorld()
gamma = 0.9

### 练习 1：用采样估计 q 值

从状态 `s`、先执行动作 `a`、之后按确定性策略 `pi`（形状 `(S,)`）走 `length` 步，返回折扣回报。

In [ ]:
def sample_return(env, s, a, pi, gamma, length):
    env.reset(int(s))
    # TODO: 先执行 a；之后每一步执行 pi[当前状态]；累加 γ^t r
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def sample_return(env, s, a, pi, gamma, length):
    env.reset(int(s))
    G, disc = 0.0, 1.0
    action = a
    for _ in range(length):
        ns, r = env.step(action)
        G += disc * r
        disc *= gamma
        action = pi[ns]
    return G

In [ ]:
# ✅ 自测 + MC Basic（网格世界是确定性的，所以每个 (s,a) 采 1 条轨迹就够了）
def mc_basic(env, gamma, length, n_iter=10):
    pi = np.zeros(env.n_states, dtype=int)
    for k in range(n_iter):
        q = np.array([[sample_return(env, s, a, pi, gamma, length) for a in range(5)] for s in range(25)])
        pi = q.argmax(1)
    return q, pi

q_mc, pi_mc = mc_basic(env, gamma, length=100)
# 与有模型的值迭代结果对比
v = np.zeros(25)
for _ in range(500):
    v = (env.R + gamma * env.P @ v).max(1)
assert np.allclose(q_mc.max(1), v, atol=1e-3)
print("通过 ✅ MC Basic 找到了最优值")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, L in zip(axes, [1, 3, 15]):
    q_L, pi_L = mc_basic(env, gamma, length=L)
    env.plot(V=q_L.max(1), policy=pi_L, title=f"episode length = {L}", ax=ax)
plt.tight_layout(); plt.show()

观察：**轨迹长度很重要**。长度太短时，远离目标的格子"看不到"目标的奖励，值为 0、策略乱选。只有轨迹足够长，奖励信号才能传播到所有状态。这对真机启示很大：**任务越长、奖励越稀疏，RL 越难**。

---

## 2. 提高数据效率：每一次访问都利用起来

MC Basic 很浪费：一条轨迹 $s_1\xrightarrow{a_2}s_2\xrightarrow{a_4}s_1\xrightarrow{a_2}s_2\to\cdots$ 只用来估计**起点**的 $q(s_1,a_2)$。实际上它的**每个后缀**都是一条从中间某个 $(s,a)$ 出发的轨迹。

- **first-visit**：每条轨迹里只用每个 $(s,a)$ 第一次出现时的回报；
- **every-visit**：每次出现都用。

从后往前遍历轨迹，用 $G\leftarrow\gamma G + r_{t+1}$ 就能一次性算出所有后缀的回报。

## 3. 去掉"探索性起点"：ε-greedy 策略

如果只从固定的起点出发，按确定性策略走，很多 $(s,a)$ 永远访问不到，它们的 $q$ 永远是错的。两个解决办法：

1. **探索性起点（Exploring Starts）**：每个 $(s,a)$ 都当过起点。真机上做不到，你不能让机器人从任意状态开始。
2. **软策略（soft policy）**：让每个动作都有一定概率被选到。最常用的是 **ε-greedy**：

$$
\pi(a|s)=\begin{cases}1-\varepsilon+\frac{\varepsilon}{|\mathcal A|} & a=\arg\max_{a'}q(s,a')\\[2pt] \frac{\varepsilon}{|\mathcal A|}&\text{其它动作}\end{cases}
$$

$\varepsilon$ 平衡**探索**（exploration）和**利用**（exploitation）：$\varepsilon$ 越大探索越多，但策略本身离最优越远。

### 练习 2：ε-greedy 策略矩阵

In [ ]:
def epsilon_greedy(q, eps):
    """q: (S, A) -> 策略矩阵 (S, A)"""
    # TODO: 每行先全部填 eps/A，再在 argmax 位置加上 1-eps
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def epsilon_greedy(q, eps):
    """q: (S, A) -> 策略矩阵 (S, A)"""
    S, A = q.shape
    pi = np.full((S, A), eps / A)
    pi[np.arange(S), q.argmax(1)] += 1 - eps
    return pi

In [ ]:
# ✅ 自测
p = epsilon_greedy(np.array([[1.0, 3.0, 2.0]]), 0.3)
assert np.allclose(p, [[0.1, 0.8, 0.1]]) and np.allclose(p.sum(1), 1)
print("通过 ✅")

### 练习 3：MC ε-greedy（every-visit）

In [ ]:
def mc_epsilon_greedy(env, gamma, eps, n_episodes=300, length=200, seed=0):
    rng = np.random.default_rng(seed)
    q = np.zeros((env.n_states, env.n_actions))
    returns_sum = np.zeros_like(q)
    returns_cnt = np.zeros_like(q)
    for ep in range(n_episodes):
        pi = epsilon_greedy(q, eps)
        # 1) 从随机状态出发，按 pi 采样一条长度为 length 的轨迹
        s = env.reset(int(rng.integers(env.n_states)))
        traj = []
        for _ in range(length):
            a = rng.choice(env.n_actions, p=pi[s])
            ns, r = env.step(a)
            traj.append((s, a, r))
            s = ns
        # 2) TODO: 从后往前遍历 traj，G = γG + r；对每个 (s,a) 累加 returns_sum、returns_cnt，
        #          并更新 q[s,a] = returns_sum / returns_cnt（every-visit）
        raise NotImplementedError
    return q

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def mc_epsilon_greedy(env, gamma, eps, n_episodes=300, length=200, seed=0):
    rng = np.random.default_rng(seed)
    q = np.zeros((env.n_states, env.n_actions))
    returns_sum = np.zeros_like(q)
    returns_cnt = np.zeros_like(q)
    for ep in range(n_episodes):
        pi = epsilon_greedy(q, eps)
        s = env.reset(int(rng.integers(env.n_states)))
        traj = []
        for _ in range(length):
            a = rng.choice(env.n_actions, p=pi[s])
            ns, r = env.step(a)
            traj.append((s, a, r))
            s = ns
        G = 0.0
        for s_t, a_t, r_t in reversed(traj):
            G = gamma * G + r_t
            returns_sum[s_t, a_t] += G
            returns_cnt[s_t, a_t] += 1
            q[s_t, a_t] = returns_sum[s_t, a_t] / returns_cnt[s_t, a_t]
    return q

In [ ]:
# ✅ 自测：比较不同的 ε
def greedy_value(q):
    """评估"对 q 取贪心"得到的确定性策略的真实值"""
    pi_m = policy_to_matrix(q.argmax(1))
    return np.linalg.solve(np.eye(25) - gamma * np.einsum("sa,sat->st", pi_m, env.P), (pi_m * env.R).sum(1))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
gaps = {}
for ax, eps in zip(axes, [0.1, 0.5, 0.9]):
    q = mc_epsilon_greedy(env, gamma, eps, n_episodes=1000)
    v_g = greedy_value(q)
    gaps[eps] = np.abs(v_g - v).mean()
    env.plot(V=v_g, policy=q.argmax(1), title=f"eps={eps}  mean gap to v*={gaps[eps]:.2f}", ax=ax)
    print(f"eps={eps}: 贪心策略与最优值的平均差距 {gaps[eps]:.3f}")
plt.tight_layout(); plt.show()
assert gaps[0.5] < 0.3
print("通过 ✅")

观察（不同随机种子结果会有波动）：

- **ε 太小**（0.1）：探索不足。一旦某条路"还不错"，智能体就一直走它，其它更好的路很少被尝试，有些状态学得很差。
- **ε 适中**（0.5）：探索充分，贪心策略接近最优。
- **ε 太大**（0.9）：数据几乎是随机游走，学到的是"随机性很强的策略"自己的 $q$，贪心之后也未必最优。

> 注意：$\varepsilon$ 较大时，学到的 $q$ 是 **ε-greedy 策略本身**的动作值。因为它"随时可能乱走一步"，所以会更"保守"，倾向于远离禁区。这叫 on-policy 学习的"安全性偏好"。下一课的 Q-learning（off-policy）没有这个问题。

---

## 小结

| 方法 | 思想 | 缺点 |
|---|---|---|
| MC Basic | 用平均回报代替模型计算 $q$ | 数据效率极低 |
| every-visit MC | 一条轨迹的所有后缀都用上 | 需要等回合结束 |
| MC ε-greedy | 用软策略保证探索 | $\varepsilon$ 让策略不是最优 |

MC 的共同缺点：**必须等一整条轨迹结束才能更新**，而且回报的方差大（随机性一路累积）。下一课先学一个数学工具（随机近似），然后在 2.07 用 TD 方法解决这两个问题。